# VERITAS v0.2.0 · CPU evidence verification
Verify a NASA excerpt, change the number of independent sources, and follow an English claim through a Spanish count and a second cited document. Run every cell in order. The controlled benchmark runs locally in the notebook. A separate cell reads the published SciFact measurements.

In [ ]:
import os
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path

release = "v0.2.0"
local = next((p for p in [Path.cwd(), Path.cwd().parent] if (p / "src/veritas").is_dir()), None)
if Path("/content").is_dir() or local is None:
    root = (Path("/content") if Path("/content").is_dir() else Path.cwd()) / f"veritas-{release}"
    if not root.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                release,
                "https://github.com/Rahulchaube1/veritas.git",
                str(root),
            ],
            check=True,
        )

    def revision(ref):
        return subprocess.check_output(["git", "-C", str(root), "rev-parse", ref], text=True).strip()

    if revision("HEAD") != revision(f"{release}^{{commit}}"):
        raise RuntimeError(f"The notebook requires the {release} checkout at {root}")
else:
    root = local
loaded = sys.modules.get("veritas")
if loaded and Path(loaded.__file__).resolve().parent != (root / "src/veritas").resolve():
    raise RuntimeError("A different VERITAS checkout is loaded. Restart the session and run all cells.")
os.chdir(root)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
sys.path.insert(0, str(root / "src"))
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
print("VERITAS", version("veritas-evidence"), root)
assert version("veritas-evidence") == release[1:]

In [ ]:
import json

from veritas import Veritas

engine = Veritas()
engine.add([json.loads(line) for line in Path("datasets/public_examples.jsonl").read_text().splitlines()])
result = engine.search("Neptune is the eighth and most distant planet in our solar system.")
print(result.answer)
for i in result.claims[0].citations:
    passage = result.evidence[i]
    print(passage.passage, passage.url)
assert result.claims[0].verdict == "supported"

## Duplicate injection
The same opposing report appears on eight nominally different pages. Compare the two aggregation rules on the same retrieved documents.

In [ ]:
from veritas import Config

copies = Veritas(Config(page_size=12))
claim = "The ferry carries 40 passengers."
copies.add(
    [{"id": "primary", "text": claim}]
    + [{"id": f"copy-{n}", "text": "The ferry carries 41 passengers."} for n in range(8)]
)
for grouped in (True, False):
    r = copies.search(claim, dependency=grouped)
    print("grouped", grouped, "verdict", r.claims[0].verdict, "balance", round(r.evidence_balance, 3))
assert copies.search(claim).claims[0].verdict == "conflict"

## Check a two-document inference
Which hospital does Cedar Trust operate, and how many beds does that hospital have? Both source spans must support their respective steps. Try changing the requested facility from hospital to depot.

In [ ]:
joined = Veritas()
joined.add(
    [
        {"id": "register", "text": "Cedar Trust operates Hospital Mar."},
        {"id": "survey", "text": "El hospital Mar tiene 18 camas."},
    ]
)
for kind in ["hospital", "depot"]:
    result = joined.search(f"The {kind} operated by Cedar Trust has 18 beds.")
    decision = result.claims[0]
    print(kind, decision.verdict)
    for i in decision.citations:
        e = result.evidence[i]
        print(" ", e.doc_id, (e.start, e.end), e.passage)
    assert decision.verdict == ("supported" if kind == "hospital" else "insufficient")
    if kind == "hospital":
        assert {result.evidence[i].doc_id for i in decision.citations} == {"register", "survey"}
joined.close()

## Reproduce the synthetic benchmark
This cell regenerates all 1,440 cases, fits the synthetic calibrator and scores the 360 designated test cases.

In [ ]:
output_root = Path(".cache/colab-run")
subprocess.run([sys.executable, "experiments/controlled.py", "--output-root", str(output_root)], check=True)
metrics = json.loads((output_root / "results/controlled_v2.json").read_text())
print("Original 360 cases", metrics["ablations"]["full"])
print("Per-family evidence verdict accuracy", metrics["families"])
print("Separate type-corrected cases", metrics["type_corrected_two_hop"])
assert round(metrics["worlds"] * metrics["ablations"]["full"]["verdict_accuracy"]) == 330
assert metrics["type_corrected_two_hop"]["correct_with_both_documents"] == 30

## Inspect the published SciFact comparison
Both profiles had the same three-document budget for each of 172 claims. They examined the same documents per claim. The following cell reads the recorded results. Rebuilding both runs requires the SciFact archive and optional neural checkpoints. [Evaluation protocol](https://github.com/Rahulchaube1/veritas/blob/v0.2.0/docs/verification-v0.2.md)

In [ ]:
audit = json.loads(Path("results/scientific_172_summary.json").read_text())
for profile in ["baseline", "revised"]:
    scores = audit[profile]
    print(
        profile,
        "| gold verdicts",
        scores["correct_all"],
        "/",
        scores["queries"],
        "| correct decisive",
        scores["correct_decisive"],
        "| wrong decisive",
        scores["wrong_decisive"],
        "| correct with a matching citation",
        scores["grounded_correct"],
    )

## Open the search interface in Colab
Set LAUNCH_UI to True in a Colab runtime. The application runs in that runtime and uses the two public excerpts already loaded above.

In [ ]:
LAUNCH_UI = False
if LAUNCH_UI:
    import threading

    import uvicorn
    from google.colab import output

    from veritas.api import create_app

    app = create_app(engine)
    threading.Thread(
        target=uvicorn.run, args=(app,), kwargs={"host": "127.0.0.1", "port": 8000}, daemon=True
    ).start()
    output.serve_kernel_port_as_iframe(8000, height=760)

The reproducibility guide covers the official SciFact archive, optional pretrained models and learned scientific rationale filter. The default CPU demonstration does not evaluate open-domain factual accuracy.